# Arize AI Phoenix Hallucination Evaluation

The purpose of this notebook is to run test cases of hallucination evaluations on datasets.

In [39]:
import os
import json
import pandas as pd
import numpy as np
import random
import sys
import io
import copy
import anthropic
from tqdm import tqdm
from dotenv import load_dotenv
import concurrent.futures
import time
import traceback
import copy
import nest_asyncio
from phoenix.evals import HallucinationEvaluator, run_evals, AnthropicModel
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## LLM Link

In [40]:
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_KEY")

In [41]:
# set up the claude model
claude_model = "claude-3-5-haiku-20241022"

## Data

In [42]:
# load json dataset
file_path = "../../data/Hallucination/qa_data.json"
with open(file_path, "r", encoding="utf-8") as file:
    data = [json.loads(line) for line in file]

In [45]:
random.seed(42)

records = []
hallucination_flags = []
answers = []

# Sample 1000 random entries from data
sampled_entries = random.sample(data, 1000)

for entry in sampled_entries:
    is_hallucinated = random.choice([True, False])
    actual_output = entry["hallucinated_answer"] if is_hallucinated else entry["right_answer"]    
    records.append({
        "input": entry["question"],
        "actual_output": actual_output,
        "context": entry["knowledge"]
    })
    hallucination_flags.append(1 if is_hallucinated else 0)
    answers.append(actual_output)

# Convert to DataFrame
df = pd.DataFrame(records).reset_index()

hallucination_df = pd.DataFrame({
    "index": df["index"],
    "hallucinated": hallucination_flags,
    "actual_output": answers
})

In [46]:
hallucination_df.hallucinated.value_counts()

hallucinated
0    534
1    466
Name: count, dtype: int64

In [47]:
arize_df = pd.DataFrame({
    'reference': df.context,
    'input': df.input,
    'output': df.actual_output,
    'context': df.context
})

## Arize AI Phoenix Evaluation

In [48]:
import nest_asyncio
import os
from phoenix.evals import HallucinationEvaluator, run_evals, AnthropicModel

# Needed for concurrency in notebook environments
nest_asyncio.apply()

# Get Anthropic API key from environment variable
os.environ["ANTHROPIC_API_KEY"] = anthropic_key

# Set up Claude model for evaluation with API key
eval_model = AnthropicModel(model=claude_model)

# Define your evaluators
hallucination_evaluator = HallucinationEvaluator(eval_model)

# Run the evaluators, each evaluator will return a dataframe with evaluation results
qa_hallucination_eval = run_evals(
    dataframe=arize_df, 
    evaluators=[hallucination_evaluator], 
    provide_explanation=True
)

run_evals |██████████| 1000/1000 (100.0%) | ⏳ 05:27<00:00 |  2.81it/s

In [49]:
# Step 1: Get results
pred_df = qa_hallucination_eval[0].reset_index()

# Step 2: Join with ground truth labels
merged_df = pd.merge(pred_df, hallucination_df[["index", "hallucinated", "actual_output"]], on="index")

# Step 3: Round scores to get predicted labels
merged_df["pred_label"] = merged_df["score"].round().astype(int)
merged_df["true_label"] = merged_df["hallucinated"].astype(int)

# Step 4: Calculate metrics
accuracy = accuracy_score(merged_df["true_label"], merged_df["pred_label"])
precision = precision_score(merged_df["true_label"], merged_df["pred_label"], zero_division=0)
recall = recall_score(merged_df["true_label"], merged_df["pred_label"], zero_division=0)
f1 = f1_score(merged_df["true_label"], merged_df["pred_label"], zero_division=0)

# Output
print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")

Accuracy: 0.852
Precision: 0.906
Recall: 0.762
F1 Score: 0.828


In [50]:
merged_df.to_csv('../../results/Hallucination/Arize/qa_results.csv')